In [2]:
import pandas as pd
import re
import nltk
from nltk.corpus import wordnet
from nltk.stem.wordnet import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
df = pd.read_csv('IMDB.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [ ]:
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger_eng')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [9]:
def get_corpus(text : str) -> list:
    """ Функція для перетворення тексту в корпус.
    Під час перетворення функція чистить текст.
    param:
        text: текст для очищення
    return:
        list : корпус з очищеним текстом
    """
    text = text.lower()

    # Видалення HTML тегів
    text = re.sub(r'<.*?>', '', text)

    # Видалення всього, що не є літерами або знаками закінчення
    text = re.sub(r'[^a-zA-Z0-9 .!?]', '', text)

    # Об'єднання пробілів
    text = re.sub(r' +', ' ', text).strip()

    # Перетворення секвенцію знаків закінчення на один знак
    text = re.sub(r'[.?!]+', '.', text).strip()

    # Розбиття на документи
    corpus = re.split(r'[.!?]\s+', text)

    # Видалення з останнього документу знак закінчення
    corpus[-1] = re.sub(r'[.!?]', '', corpus[-1])

    return corpus

In [4]:
def get_wordnet_pos(treebank_tag : str) -> str:
    """ Функція конвертує теги отриманні з функції nltk.pos_tag на підходящі для WordNetLemmatizer
    param:
        treebank_tag: тег отриманий з nltk.pos_tag
    return:
        char : тег для WordNetLemmatizer
    """
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN

In [8]:
def lemmatize_corpus(corpus : list) -> list:
    """ Функція для лемматізації слів в документах корпуса
    param:
        corpus : список документів
    return:
        Відредактований corpus array
    """

    lemmatizer = WordNetLemmatizer()
    lemmatized_corpus = []    # Новий корпус

    for doc in corpus:
        split_doc = doc.split()
        # Отримуємо теги для кожного слова
        pos_tags = nltk.pos_tag(split_doc)
        dokument_words = [] # Список слів документу

        for word, tag in pos_tags:
            # Отримання тега для WordNetLemmatizer
            wordnet_tag = get_wordnet_pos(tag)
            dokument_words.append(lemmatizer.lemmatize(word, wordnet_tag))
        # Конвертуємо обратно в стрінг
        dokument = ' '.join(dokument_words)
        lemmatized_corpus.append(dokument)
    return lemmatized_corpus

In [10]:
def preprocess_review(text : str) -> str :
    """ Функціy для підготовки текста для методів безконтекстового NLP
    Param:
        text : текст для підготовки
    Return:
        str : підготовлений текст"""
    sentences = get_corpus(text)
    lemmatized_sentences = lemmatize_corpus(sentences)
    return ' '.join(lemmatized_sentences)

Підготую данні

In [ ]:
df['review'] = df['review'].apply(preprocess_review)

In [ ]:
df.head()

,review,sentiment
0,one of the other reviewer have mention that af...,positive
1,a wonderful little production the filming tech...,positive
2,i think this be a wonderful way to spend time ...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money be a ...,positive


Підготовка зайняла 9 хв, але все вийшло

Перетворимо текст таргету у int

In [ ]:
y = df['sentiment'].map({'positive': 1, 'negative': 0})

Спочатку спробую Bag of Word

In [ ]:
vectorizer = CountVectorizer(max_features=5000)
X = vectorizer.fit_transform(df['review'])

Розіб'ю датасет на тренуальну частину та натренерую логістичний регрессор

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

classifier = LogisticRegression(max_iter=1000)
classifier.fit(X_train, y_train)

y_pred = classifier.predict(X_test)

In [ ]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      0.87      0.87      4961
           1       0.87      0.88      0.88      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



Результати дуже непогані, тепер спробую TFIDF

In [ ]:
vectorizer_tfidf= TfidfVectorizer(max_features=5000)
X_tfidf = vectorizer_tfidf.fit_transform(df['review'])

In [ ]:
X_train_tfidf, X_test_tfidf, y_train_tfidf, y_test_tfidf = train_test_split(X_tfidf, y, test_size=0.2, random_state=42)

classifier_tfidf = LogisticRegression(max_iter=1000)
classifier_tfidf.fit(X_train_tfidf, y_train_tfidf)

y_pred_tfidf = classifier_tfidf.predict(X_test)

In [ ]:
print(classification_report(y_test_tfidf, y_pred_tfidf))

              precision    recall  f1-score   support

           0       0.93      0.76      0.83      4961
           1       0.80      0.94      0.86      5039

    accuracy                           0.85     10000
   macro avg       0.86      0.85      0.85     10000
weighted avg       0.86      0.85      0.85     10000



TFIDF у середньому вийшов трішки гірше, але обидва методи працюють дуже гарно